# 📈 Hands-On: Zeitreihenprognose mit Chronos
### Zero-Shot Forecasting zum Selbermachen (~15 min)

Gleiches Format wie das Prophet-Hands-on: **Setup, Daten und Auswertung sind vorgegeben** (einfach ausführen). Du schreibst nur den Modell-Teil.

| Aufgabe | Thema | Chronos-Superkraft | Zeit |
|---|---|---|---|
| 1 | Naiver Zero-Shot-Forecast | kein Training nötig | ~4 min |
| 2 | Black Friday als Kovariate | Zusatzwissen ohne Feature-Engineering | ~5 min |
| 3 | 3 Filialen in **einem** Aufruf | Panel / Cross-Learning | ~5 min |

Datensatz: Verkäufe der **„SkyDrive X1 Pro" SSD** (synthetisch: Trend + Wochenmuster + Black-Friday-Peak) – ab Aufgabe 3 in drei Filialen.

**So arbeitest du:** Alle Zellen von oben nach unten ausführen. Selbst schreiben musst du **nur in den 3 Zellen mit dem ✏️-Rahmen** (eine pro Aufgabe) – Lücken sind mit `___` bzw. `...` markiert. Alles andere ist vorgegeben.


## ⚙️ Setup – Pakete installieren & Modelle laden *(einmalig – einfach ausführen)*

Die nächste Zelle installiert die benötigten Pakete direkt aus dem Notebook, die übernächste lädt die beiden Chronos-Modelle herunter (mehrere hundert MB – beim ersten Mal ein paar Minuten, danach kommen sie aus dem Cache).

**Wenn unten ✅ erscheint, bist du startklar.** Falls nach der Installation ein Import-Fehler auftaucht: einmal den Kernel neu starten (Kernel → Restart) und die Zellen erneut ausführen.


In [ ]:
# Benötigte Pakete installieren (einmalig; beim 1. Mal 1-2 Minuten)
%pip install -q chronos-forecasting scikit-learn
print('Installation ok – weiter mit der nächsten Zelle.')


In [ ]:
# Modelle einmal herunterladen + Mini-Test.
# Beim 1. Mal: einige Minuten (Download). Danach: wenige Sekunden.
import torch
import pandas as pd
from chronos import BaseChronosPipeline, Chronos2Pipeline

pipe_warm = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-small', device_map='cpu')
_ = pipe_warm.predict_quantiles(inputs=torch.tensor([1.0] * 60), prediction_length=3,
                                quantile_levels=[0.5])

pipe2_warm = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cpu')
ctx_warm = pd.DataFrame({'id': 'test',
                         'timestamp': pd.date_range('2024-01-01', periods=60, freq='D'),
                         'target': [float(i) for i in range(60)]})
_ = pipe2_warm.predict_df(ctx_warm, prediction_length=3, quantile_levels=[0.5],
                          id_column='id', timestamp_column='timestamp', target='target')
del pipe_warm, pipe2_warm
print('✅ Setup ok – Installation funktioniert, beide Modelle im Cache. Du bist startklar!')


---
## 0 · Setup *(vorgegeben – einfach ausführen)*


In [ ]:
# (Pakete wurden oben im Setup-Schritt installiert)
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from chronos import BaseChronosPipeline
np.random.seed(0); torch.manual_seed(0)

def print_metrics(y_true, y_pred, label='Modell'):
    y_true = np.asarray(y_true, float); y_pred = np.asarray(y_pred, float)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    print(f'{label:<28} RMSE={rmse:8.1f}  MAE={mae:8.1f}  MAPE={mape:5.1f}%')
    return {'label': label, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

def plot_forecast(y_train, y_true, median, lo, hi, label, n_context=90):
    H = len(y_true)
    xh = np.arange(-min(n_context, len(y_train)), 0); xf = np.arange(0, H)
    plt.figure(figsize=(13, 4))
    plt.plot(xh, y_train[-len(xh):], color='gray', label='Historie')
    plt.plot(xf, y_true, color='black', marker='.', label='Test (echt)')
    plt.plot(xf, median, color='C1', label=f'{label} - Median')
    plt.fill_between(xf, lo, hi, color='C1', alpha=0.25, label='10-90%-Band')
    plt.axvline(0, color='k', ls=':'); plt.legend(); plt.title(label)
    plt.tight_layout(); plt.show()


## 0.1 · Datengenerierung *(vorgegeben – einfach ausführen)*

Erzeugt die SkyDrive-Zeitreihe (Trend + Wochenmuster + Black-Friday-Spitze) und teilt in Train/Test. Der Test-Zeitraum enthält bewusst einen **Black Friday** – darauf zielt Aufgabe 2.


In [ ]:
# ============================================================
# DATEN  —  IDENTISCH zur Prophet-Handson-Übung
# Gleicher Seed und gleiche Reihenfolge -> exakt dieselben y-Werte.
# (eine saubere Black-Friday-Spitze pro Jahr)
# ============================================================

# --- 1) Basis-Bausteine (in USD) -----------------------------------
np.random.seed(42)
dates = pd.date_range(start='2020-01-01', end='2023-12-31')   # 4 Jahre wie Prophet
df = pd.DataFrame({'ds': dates})

df['trend']  = np.linspace(6450, 19350, len(dates))
df['noise']  = np.random.normal(0, 650, len(dates))
df['weekly'] = df['ds'].dt.dayofweek.map(
    {0:-1030, 1:-1290, 2:-1290, 3:-1030, 4:650, 5:2580, 6:1935})
df['yearly'] = 3225 * np.sin(2*np.pi*(df['ds'].dt.dayofyear-100)/365.25)

# --- 2) Black-Friday-Effekt (Tage -3..+1 zusammenhaengend -> EINE Spitze)
def black_friday(year):
    return pd.date_range(f'{year}-11-01', f'{year}-11-30', freq='W-FRI')[3]

black_fridays = pd.to_datetime([black_friday(y) for y in [2020, 2021, 2022, 2023]])

rng = np.random.default_rng(seed=42)
df['bf_effect'] = 0.0
for bf in black_fridays:
    year_factor = np.clip(rng.normal(1.0, 0.2), 0.6, 1.4)
    for offset, extra in [(-3, 18000), (-2, 22000), (-1, 30000), (0, 65000), (1, 35000)]:
        day_noise = rng.normal(0, extra * 0.10)
        scaled    = extra * year_factor + day_noise
        df.loc[df['ds'] == bf + pd.Timedelta(days=offset), 'bf_effect'] = max(0, scaled)

# --- 3) Zielvariable y ---------------------------------------------
df['y'] = df['trend'] + df['noise'] + df['weekly'] + df['yearly'] + df['bf_effect']

# --- 4) Black-Friday-Flag (0/1) als Kovariate fuer Chronos-2 -------
# Gleiches Fenster wie Prophets bf_df (lower_window=-3, upper_window=+1)
df['black_friday'] = 0.0
for bf in black_fridays:
    win = pd.date_range(bf - pd.Timedelta(days=3), bf + pd.Timedelta(days=1))
    df.loc[df['ds'].isin(win), 'black_friday'] = 1.0

# --- 5) Train/Test-Split: letzte H Tage (enthaelt Black Friday 2023)
H = 60
y = df['y'].to_numpy(float)
y_train, y_test = y[:-H], y[-H:]

print(f'{len(df)} Tage von {df["ds"].min().date()} bis {df["ds"].max().date()}')
print(f'Train: {len(y_train)} Tage  |  Test (H): {H} Tage')
print('Test-Zeitraum:', df['ds'].iloc[-H].date(), 'bis', df['ds'].iloc[-1].date())
print(f'Black-Friday-Flag aktiv an {int(df["black_friday"].sum())} Tagen')


## 0.2 · Daten visualisieren *(vorgegeben – einfach ausführen)*


In [ ]:
plt.figure(figsize=(13, 4))
plt.plot(df['ds'], df['y'], color='C0', lw=1)
bf = df[df['black_friday'] == 1]
plt.scatter(bf['ds'], bf['y'], color='red', zorder=5, label='Black Friday')
plt.title('SkyDrive X1 Pro - Tagesumsatz'); plt.legend(); plt.tight_layout(); plt.show()


---
## 🎯 Aufgabe 1 – Naiver Zero-Shot-Forecast (~4 min)

Lade ein Chronos-Bolt-Modell und sage die Testperiode vorher – **ohne jede Zusatzinfo**. Gegenstück zum „naiven Prophet-Modell ohne Feiertage".

**Schritte:**
1. Modell laden: `BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-small', device_map='cpu')`
2. Vorhersagen: `predict_quantiles(inputs=torch.tensor(y_train, dtype=torch.float32), prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])` (das erste Argument heißt `inputs`)
3. Ergebnisse herausziehen – die Reihenfolge folgt `quantile_levels=[0.1, 0.5, 0.9]`:
   - `lo1     = quantiles[0, :, 0].numpy()`   *(0.1-Quantil, unteres Band)*
   - `median1 = quantiles[0, :, 1].numpy()`   *(0.5 = Median, die Prognose)*
   - `hi1     = quantiles[0, :, 2].numpy()`   *(0.9-Quantil, oberes Band)*
4. Bewerten: `print_metrics(y_test, median1, 'Aufgabe 1: naiv')`


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  ✏️✏️✏️  AUFGABE 1 – DEIN CODE KOMMT HIER REIN  ✏️✏️✏️  ║
# ╚══════════════════════════════════════════════════════════╝
# pipe = ...
# quantiles, mean = pipe.predict_quantiles(...)
# lo1, median1, hi1 = ...   # Reihenfolge = quantile_levels [0.1, 0.5, 0.9]!
# print_metrics(y_test, median1, 'Aufgabe 1: naiv')


## 📊 Ergebnis Aufgabe 1 visualisieren *(vorgegeben – einfach ausführen)*


In [ ]:
plot_forecast(y_train, y_test, median1, lo1, hi1, 'Aufgabe 1: naiv (zero-shot)')


---
## 🎯 Aufgabe 2 – Chronos-2 mit Black-Friday-Kovariate (~5 min)

Bolt verpasst den Black Friday – obwohl er **dreimal in der Historie** steckt! Jetzt legen wir **zwei Hebel gleichzeitig** um: das neuere Modell **Chronos-2** und sein Kovariaten-Feature – wir sagen dem Modell zusätzlich, *wann* Black Friday ist (Spalte `df['black_friday']` ist schon vorhanden). Gegenstück zum „Experten-Modell" bei Prophet – nur ohne `fit()` und ohne Feature-Engineering. *Welcher der beiden Hebel wie viel bringt, klärt die Musterlösung – die Antwort überrascht!*

Kopiere die Skizze in die Zelle darunter, fülle die Lücken `___` und speichere **`median2`**, **`lo2`**, **`hi2`**:

```python
from chronos import Chronos2Pipeline
pipe2 = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cpu')

context_df = pd.DataFrame({'id': 'skydrive',
                           'timestamp': df['ds'][:-H].values,
                           'target': ___,                              # <- Trainingswerte
                           'black_friday': df['black_friday'].to_numpy()[:-H]})
future_df  = pd.DataFrame({'id': 'skydrive',
                           'timestamp': df['ds'][-H:].values,
                           'black_friday': ___})                       # <- Kovariate im Testzeitraum
pred_df = pipe2.predict_df(context_df, future_df=___, prediction_length=H,
                           quantile_levels=[0.1, 0.5, 0.9],
                           id_column='id', timestamp_column='timestamp', target='target')
lo2, median2, hi2 = (pred_df[q].to_numpy() for q in ['0.1', '___', '0.9'])
```


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  ✏️✏️✏️  AUFGABE 2 – DEIN CODE KOMMT HIER REIN  ✏️✏️✏️  ║
# ╚══════════════════════════════════════════════════════════╝
# (Skizze aus der Zelle darüber kopieren & Lücken füllen)
# median2, lo2, hi2 = ...
# print_metrics(y_test, median2, 'Aufgabe 2: Chronos-2 + Black Friday')


## 📊 Showdown: naiv vs. verbessert *(vorgegeben – einfach ausführen)*


In [ ]:
print_metrics(y_test, median1, 'Aufgabe 1: naiv')
print_metrics(y_test, median2, 'Aufgabe 2: verbessert')

xf = np.arange(H)
plt.figure(figsize=(13, 4))
plt.plot(xf, y_test, color='black', marker='.', label='Test (echt)')
plt.plot(xf, median1, color='C1', label='naiv')
plt.plot(xf, median2, color='C0', label='verbessert')
plt.title('Showdown: naiv vs. verbessert'); plt.legend(); plt.tight_layout(); plt.show()


---
## 🎯 Aufgabe 3 – Drei Filialen, EIN Aufruf (Panel-Forecast, ~5 min)

Die SkyDrive wird in **drei Filialen** verkauft. `Hamburg` hat erst vor **gut einem Jahr** eröffnet – und damit erst **einen einzigen** Black Friday erlebt!

**Chronos-2 kann alle Serien in EINER Tabelle verarbeiten** (Panel): ein Aufruf für alle Filialen. Die Kovariate sagt, *wann* Black Friday ist – und für die junge Filiale reicht ihr **eines** Beispiel im Kontext, damit Chronos-2 versteht, was das Flag bedeutet.

**Deine Aufgabe (2 Schritte):**
1. Staple die vorbereiteten Filial-Tabellen zu **einem** Panel: `context_mv = pd.concat(ctx_list, ignore_index=True)` – genauso `future_mv` aus `fut_list`.
2. **Ein** `predict_df`-Aufruf für alle drei Filialen (gleiche Argumente wie in Aufgabe 2, `pipe2` wiederverwenden) → Ergebnis als **`pred_mv`** speichern.


### 3.0 · Filialdaten erzeugen *(vorgegeben – einfach ausführen)*


In [ ]:
# ============================================================
# AUFGABE-3-DATEN (vorgegeben) — 3 Filialen, gleiche Muster, andere Größe
# 'Hamburg' hat erst vor gut einem Jahr eröffnet -> genau EIN Black Friday in der Historie!
# ============================================================
from chronos import Chronos2Pipeline
if 'pipe2' not in globals():                     # falls Aufgabe 2 übersprungen wurde
    pipe2 = Chronos2Pipeline.from_pretrained('amazon/chronos-2', device_map='cpu')

rng3 = np.random.default_rng(seed=7)
base = (df['trend'] + df['weekly'] + df['yearly'] + df['bf_effect']).to_numpy()

factors = {'Berlin': 1.00, 'München': 1.35, 'Hamburg': 0.75}
start   = {'Berlin': 0,    'München': 0,    'Hamburg': len(base) - (400 + H)}  # ~13 Monate (enthält den BF 2022)

stores = {name: f * base + rng3.normal(0, 650, len(base)) for name, f in factors.items()}

# Pro Filiale: Kontext-Tabelle (Vergangenheit MIT Kovariate) + Future-Tabelle (bekannte Kovariaten-Zukunft)
ctx_list, fut_list = [], []
for name in factors:
    s0 = start[name]
    ctx_list.append(pd.DataFrame({'id': name,
                                  'timestamp': df['ds'].iloc[s0:-H].values,
                                  'target': stores[name][s0:-H],
                                  'black_friday': df['black_friday'].to_numpy()[s0:-H]}))
    fut_list.append(pd.DataFrame({'id': name,
                                  'timestamp': df['ds'].iloc[-H:].values,
                                  'black_friday': df['black_friday'].to_numpy()[-H:]}))

for name in factors:
    print(f"{name:<10} Historie: {len(stores[name][start[name]:-H]):4d} Tage | Test: {H} Tage")


In [ ]:
# Die drei Filialen ansehen (vorgegeben) — oben gesamt, unten Zoom auf den Teststart.
# Grau schattiert = Testzeitraum: Diese ECHTEN Werte (inkl. Black-Friday-Peak) sieht
# das Modell nicht — genau sie gilt es vorherzusagen.
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 7))

for name in factors:
    s0 = start[name]
    ax1.plot(df['ds'].iloc[s0:], stores[name][s0:], lw=0.8, label=name)
ax1.axvspan(df['ds'].iloc[-H], df['ds'].iloc[-1], color='gray', alpha=0.15,
            label='Testzeitraum (echte Werte – für das Modell unsichtbar)')
ax1.axvline(df['ds'].iloc[-H], color='k', ls=':')
ax1.set_title('3 Filialen — gepunktete Linie = Teststart')
ax1.legend()

zoom0 = len(df) - H - 30
for name in factors:
    ax2.plot(df['ds'].iloc[zoom0:], stores[name][zoom0:], lw=1.2, marker='.', ms=4, label=name)
ax2.axvspan(df['ds'].iloc[-H], df['ds'].iloc[-1], color='gray', alpha=0.15,
            label='Testzeitraum (echte Werte)')
ax2.axvline(df['ds'].iloc[-H], color='k', ls=':')
ax2.set_title('Zoom auf den Teststart — rechts der Linie liegen die ECHTEN Werte inkl. Black-Friday-Peak')
ax2.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ╔══════════════════════════════════════════════════════════╗
# ║  ✏️✏️✏️  AUFGABE 3 – DEIN CODE KOMMT HIER REIN  ✏️✏️✏️  ║
# ╚══════════════════════════════════════════════════════════╝
# context_mv = pd.concat(___)
# future_mv  = ...
# pred_mv    = pipe2.predict_df(...)   # EIN Aufruf, alle Filialen!
# print(pred_mv.head())


## 📊 Auswertung Aufgabe 3 *(vorgegeben – einfach ausführen)*

Diese Zelle beantwortet **eine** Frage: *Wie viel besser ist deine Panel-Prognose als ein naiver Einzelkämpfer?* Dafür braucht sie drei Zutaten:

1. **Der Gegner:** Bolt-small aus Aufgabe 1. Er bekommt jede Filiale **einzeln** und ohne Kovariate – drei getrennte Aufrufe, während dein Panel nur EINEN brauchte.
2. **Der Schiedsrichter:** Für jede Filiale werden beide Prognosen an den echten 60 Testtagen gemessen (RMSE, kleiner = besser) → Tabelle und Balkendiagramm.
3. **Die Vorhersagen im Vergleich:** Zum Schluss pro Filiale ein Bild – echte Testwerte (schwarz) gegen beide Prognosen (orange = Bolt einzeln, blau = Panel). Achte besonders auf **Hamburg**: Bolt hat in seiner kurzen Historie zwar EINEN Black Friday gesehen, kann ihn aber nicht deuten und zieht eine flache Linie. Chronos-2 kombiniert dieses **eine Beispiel** mit der **Kovariate** (die sagt: »jetzt wieder!«) – und trifft den Peak.

> **Fairness-Hinweis:** Der Vergleich ist bewusst *unfair* („naiv & einzeln" gegen „alles"), um den Gesamteffekt der Zusatzinfos zu zeigen – siehe Diskussionsfrage 3.


In [ ]:
# ============================================================
# AUSWERTUNG (vorgegeben) — Wie viel besser ist das Panel als ein Einzelkämpfer?
# Ablauf: 1) Gegner laden  2) pro Filiale beide Prognosen holen
#         3) RMSE-Tabelle  4) Balkendiagramm  5) Blick auf Hamburg
# ============================================================

# --- 1) Der GEGNER: Bolt-small aus Aufgabe 1 --------------------------------
#     Er bekommt weder Kovariate noch Panel: jede Filiale einzeln, nur Historie.
if 'pipe' not in globals():                      # nur laden, falls noch nicht da
    pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-small', device_map='cpu')

rows, bolt_med = [], {}
for name in factors:                             # 'Berlin', 'München', 'Hamburg'
    # --- 2a) Daten DIESER Filiale herausschneiden ---------------------------
    y_tr = stores[name][start[name]:-H]          # Historie (Hamburg: nur ~13 Monate!)
    y_te = stores[name][-H:]                     # echte letzte 60 Tage = "Prüfungsbogen"

    # --- 2b) Prognose des Gegners: EIN Aufruf PRO Filiale, isoliert ----------
    q, _ = pipe.predict_quantiles(inputs=torch.tensor(y_tr, dtype=torch.float32),
                                  prediction_length=H, quantile_levels=[0.1, 0.5, 0.9])
    bolt_med[name] = q[0, :, 1].numpy()          # Median merken (für den Hamburg-Plot)

    # --- 2c) DEINE Panel-Prognose aus der großen Tabelle filtern -------------
    #     pred_mv hat eine Zeile pro Filiale UND Tag -> die id-Spalte trennt sie.
    med_c2 = pred_mv.loc[pred_mv['id'] == name, '0.5'].to_numpy()   # 60 Median-Werte

    # --- 2d) Der SCHIEDSRICHTER: beide gegen die echten Werte messen ---------
    rows.append({'Filiale': name,
                 'Bolt-small (einzeln, naiv)': np.sqrt(mean_squared_error(y_te, bolt_med[name])),
                 'Chronos-2 (Panel + Kovariate)': np.sqrt(mean_squared_error(y_te, med_c2))})

# --- 3) Ergebnis-Tabelle: eine Zeile pro Filiale + Durchschnitt --------------
mv = pd.DataFrame(rows).set_index('Filiale')
mv.loc['Ø (Mittel)'] = mv.mean()
print('RMSE je Filiale (kleiner = besser)')
print(mv.round(1))

# --- 4) Dieselbe Tabelle als Bild: pro Filiale zwei Balken -------------------
#     grau = Gegner (einzeln & naiv), türkis = dein Panel + Kovariate
ax = mv.plot.bar(figsize=(9, 4), color=['#95A5A6', '#20B2AA'], rot=0)
ax.set_ylabel('RMSE'); ax.set_title('Einzeln & naiv vs. Panel + Kovariate')
plt.tight_layout(); plt.show()

# --- 5) Die Vorhersagen im Vergleich: pro Filiale echt vs. beide Prognosen ---
#     grau = letzte 30 Tage Historie | schwarz = echte Testwerte
#     orange = Bolt (einzeln, naiv)  | blau = Chronos-2 (Panel + Kovariate)
fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)
for ax, name in zip(axes, factors):
    ax.plot(df['ds'].iloc[-H-30:-H], stores[name][-H-30:-H], color='gray', lw=1,
            label='Historie (letzte 30 Tage)')
    test_x = df['ds'].iloc[-H:]
    ax.plot(test_x, stores[name][-H:], color='black', marker='.', label='Test (echt)')
    ax.plot(test_x, bolt_med[name], color='C1', lw=1.8, label='Bolt-small (einzeln)')
    ax.plot(test_x, pred_mv.loc[pred_mv['id'] == name, '0.5'].to_numpy(), color='C0', lw=2,
            label='Chronos-2 (Panel + Kovariate)')
    ax.axvline(df['ds'].iloc[-H], color='k', ls=':')
    extra = ' — junge Filiale: nur EIN Black Friday in der Historie!' if name == 'Hamburg' else ''
    ax.set_title(f'{name}{extra}')
    ax.set_ylabel('Umsatz'); ax.legend(fontsize=9, ncol=2)
plt.tight_layout(); plt.show()


---
## 💬 Diskussionsfragen

1. Chronos braucht **kein** `fit()` und keine Feiertage – warum funktioniert es trotzdem?
2. `Hamburg` hat nur EINEN Black Friday in der Historie. Warum reicht der für Chronos-2 – und warum hätte es mit 90 Tagen Historie (null Black Fridays) **nicht** funktioniert?
3. In Aufgabe 3 tritt „Bolt einzeln + naiv" gegen „Chronos-2 + alles" an. Was wäre ein *fairer* Vergleich?
4. Wann würdest du Prophet bevorzugen, wann Chronos? (Erklärbarkeit, Rechenzeit, viele Serien, Domänenwissen)
